## Imports

In [1]:
from pathlib import Path
import sys
import os

import numpy as np

import pycuda.driver as cuda
from pycuda.compiler import SourceModule

In [2]:
from common import read_file_str, show_formatted_cpp, replace_constants_in_kernel

In [3]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Parameters

In [4]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

In [5]:
from src.parameters import GRAVITY

## Load in data from mesh

In [6]:
acceleration = np.load('./profiling/numpy/piece_acceleration.npy')

## Cuda Parameters

In [7]:
BLOCK_SIZE = 1024
NR_BLOCKS = (len(acceleration) + BLOCK_SIZE - 1) // BLOCK_SIZE

## Compile cuda kernel

In [8]:
cuda_code = read_file_str("./profiling/kernels/apply_gravity.cu")

In [9]:
cuda_code = replace_constants_in_kernel(cuda_code, {"GRAVITY": GRAVITY})

In [10]:
show_formatted_cpp(cuda_code)

In [11]:
mod = SourceModule(cuda_code)

C:\Users\CYBORG\AppData\Local\Temp\ipykernel_15432\3464942708.py:1: UserWarning: The CUDA compiler succeeded, but said the following:
kernel.cu

  mod = SourceModule(cuda_code)


## Set-up memory for running kernel

In [12]:
apply_gravity = mod.get_function("apply_gravity")

In [13]:
nr_vertices = np.uint32(len(acceleration))

### Allocate memory to gpu

In [14]:
assert acceleration.flatten().flags['C_CONTIGUOUS']

In [15]:
acceleration_gpu = cuda.mem_alloc(acceleration.nbytes)

In [16]:
cuda.memcpy_htod(acceleration_gpu, acceleration.flatten())

## Profile function

In [17]:
def apply_gravity_kernel():
    apply_gravity(acceleration_gpu, nr_vertices,
                  block=(BLOCK_SIZE, 1, 1), grid=(NR_BLOCKS, 1, 1))

In [18]:
%timeit apply_gravity_kernel()

17.3 µs ± 3 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Check output is as expected

In [19]:
cuda.memcpy_dtoh(acceleration, acceleration_gpu)

In [20]:
assert np.all(acceleration[:, 0] == 0)
assert np.all(acceleration[:, 1] == -GRAVITY)
assert np.all(acceleration[:, 2] == 0)